In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 
import torch
from transformers import set_seed

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path
from vti_utils.utils import get_demos, get_hiddenstates


def extract_textual_pos_neg_exact_vti_cuda(
    model_path="liuhaotian/llava-v1.5-7b",
    model_base=None,
    conv_mode="llava_v1",
    data_file="/research/hal-afsharim/VTI/experiments/data/coco",
    demos_file="/research/hal-afsharim/VTI/experiments/data/coco/vti_pope_train.jsonl",
    num_demos=6300,
    mask_ratio=0.99,
    num_trials=50,
    seed=42,
    output_path="/research/hal-shared/vti_results",
):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Please run on a GPU machine.")

    device = torch.device("cuda:0")
    set_seed(seed)

    class Args:
        pass

    args = Args()
    args.model_path = model_path
    args.model_base = model_base
    args.conv_mode = conv_mode
    args.data_file = data_file
    args.num_demos = num_demos
    args.mask_ratio = mask_ratio
    args.num_trials = num_trials

    model_name = get_model_name_from_path(model_path)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path, model_base, model_name
    )
    model = model.to(device).eval()

    # get_demos/get_prompts already put tokenized prompts on CUDA in this repo
    input_images, input_ids = get_demos(
        args=args,
        image_processor=image_processor,
        model=model,
        tokenizer=tokenizer,
        file_path=demos_file,
        model_is_llaval=True,
    )

    # VTI-exact hidden-state extraction path
    hidden_states = get_hiddenstates(model, input_ids, input_images)

    neg_all, pos_all, diff_all = [], [], []
    for i in range(len(hidden_states)):
        neg = hidden_states[i][0].view(-1)   # hallucinated
        pos = hidden_states[i][1].view(-1)   # truthful
        diff = pos - neg                     # contrastive direction
        neg_all.append(neg)
        pos_all.append(pos)
        diff_all.append(diff)

    absolute_text_pos = torch.stack(pos_all)
    absolute_text_neg = torch.stack(neg_all)
    fit_data = torch.stack(diff_all)

    os.makedirs(output_path, exist_ok=True)
    torch.save(fit_data.cpu(), os.path.join(output_path, "pre_pca_textual_directions.pt"))
    torch.save(absolute_text_pos.cpu(), os.path.join(output_path, "absolute_textual_truthful_states.pt"))
    torch.save(absolute_text_neg.cpu(), os.path.join(output_path, "absolute_textual_hallucinated_states.pt"))

    print("Saved CUDA-extracted tensors to:", output_path)


if __name__ == "__main__":
    extract_textual_pos_neg_exact_vti_cuda()

You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.43s/it]


Saved CUDA-extracted tensors to: /research/hal-shared/vti_results


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # or "1"
import torch
import json
from transformers import set_seed

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path
from vti_utils.utils_milad import (
    get_demos_unpaired_value_or_hvalue,
    get_hiddenstates,
    hiddenstates_to_pos_neg_stacks,
)


def extract_unpaired_pos_neg_vti_cuda(
    model_path="liuhaotian/llava-v1.5-7b",
    model_base=None,
    conv_mode="llava_v1",
    data_file="/research/hal-afsharim/VTI/experiments/data/coco",
    demos_file="/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_extracted_prompt_6300.jsonl",
    num_demos=6300,
    mask_ratio=0.99,
    num_trials=50,
    seed=42,
    output_path="/research/hal-shared/vti_results_assymetric_1000",
):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available.")

    device = torch.device("cuda")
    set_seed(seed)

    class Args:
        pass

    args = Args()
    args.model_path = model_path
    args.model_base = model_base
    args.conv_mode = conv_mode
    args.data_file = data_file
    args.num_demos = num_demos
    args.mask_ratio = mask_ratio
    args.num_trials = num_trials

    model_name = get_model_name_from_path(model_path)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path, model_base, model_name
    )
    model = model.to(device).eval()

    input_images, input_ids, meta = get_demos_unpaired_value_or_hvalue(
        args=args,
        image_processor=image_processor,
        model=model,
        tokenizer=tokenizer,
        file_path=demos_file,
        model_is_llaval=True,
    )

    hidden_states = get_hiddenstates(model, input_ids, input_images)
    flags = [m["is_positive"] for m in meta]
    pos_all, neg_all = hiddenstates_to_pos_neg_stacks(hidden_states, flags)

    os.makedirs(output_path, exist_ok=True)

    def _save_stack(name, lst):
        path = os.path.join(output_path, name)
        if not lst:
            torch.save(torch.empty(0), path)
        else:
            torch.save(torch.stack(lst).cpu(), path)

    _save_stack("absolute_textual_truthful_states.pt", pos_all)
    _save_stack("absolute_textual_hallucinated_states.pt", neg_all)

    labels = torch.tensor([1 if f else 0 for f in flags], dtype=torch.long)
    torch.save(labels.cpu(), os.path.join(output_path, "demo_side_labels.pt"))

    with open(os.path.join(output_path, "demo_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved to:", output_path)
    print("  pos count:", len(pos_all), " neg count:", len(neg_all))


if __name__ == "__main__":
    extract_unpaired_pos_neg_vti_cuda()

/research/hal-afsharim/miniconda3/envs/vti/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/research/hal-afsharim/miniconda3/envs/vti/lib/python3.9/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/research/hal-afsharim/miniconda3/envs/vti/lib/python3.9/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading chec

Saved to: /research/hal-shared/vti_results_assymetric_1000
  pos count: 4976  neg count: 1284
